In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

COUNTRIES = [
    ('Ethiopia', 'ethiopia.csv'),
    ('Kenya', 'kenya.csv'),
    ('Sudan', 'sudan.csv'),
    ('Tanzania', 'tanzania.csv'),
    ('Nigeria', 'nigeria.csv'),
]

NUMERIC_COLS = ['T2M', 'T2M_MAX', 'T2M_MIN', 'T2M_RANGE', 'PRECTOTCORR', 
                'RH2M', 'WS2M', 'WS2M_MAX', 'PS', 'QV2M']

OUTLIER_COLS = ['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M', 'WS2M', 'WS2M_MAX']

OUTPUT_DIR = os.path.join('..', 'data')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Setup complete.")

Setup complete.


In [2]:
all_cleaned_dfs = []
country_stats = {}

for country_name, file_name in COUNTRIES:
    print(f"\n{'='*50}")
    print(f"PROCESSING: {country_name}")
    
    path = os.path.join('..', file_name)
    df = pd.read_csv(path)
    df['Country'] = country_name
    df['Date'] = pd.to_datetime(df['YEAR'] * 1000 + df['DOY'], format='%Y%j')
    df['Month'] = df['Date'].dt.month
    df['Year'] = df['Date'].dt.year
    
    df.replace(-999, np.nan, inplace=True)
    
    dup_count = df.duplicated().sum()
    print(f"Duplicates: {dup_count}")
    if dup_count > 0:
        df.drop_duplicates(inplace=True)
    
    missing_pct = (df.isna().sum() / len(df)) * 100
    high_missing = missing_pct[missing_pct > 5]
    if len(high_missing) > 0:
        print(f"Columns >5% missing: {list(high_missing.index)}")
    
    print(df[NUMERIC_COLS].describe().round(2))
    
    z_scores = np.abs(stats.zscore(df[OUTLIER_COLS].dropna()))
    outlier_count = (z_scores > 3).any(axis=1).sum()
    print(f"Outliers: {outlier_count}")
    
    for col in OUTLIER_COLS:
        mean_val = df[col].mean()
        std_val = df[col].std()
        df[col] = df[col].clip(lower=mean_val - 3*std_val, upper=mean_val + 3*std_val)
    
    df[OUTLIER_COLS] = df[OUTLIER_COLS].ffill()
    row_missing = df.isna().sum(axis=1) / len(df.columns)
    df = df[row_missing <= 0.3]
    
    output_file = os.path.join(OUTPUT_DIR, f"{country_name.lower()}_clean.csv")
    df.to_csv(output_file, index=False)
    print(f"Saved: {output_file}")
    
    all_cleaned_dfs.append(df)
    country_stats[country_name] = {
        'mean_temp': df['T2M'].mean(),
        'std_temp': df['T2M'].std(),
        'mean_rain': df['PRECTOTCORR'].mean(),
        'std_rain': df['PRECTOTCORR'].std(),
        'extreme_heat_days': (df['T2M_MAX'] > 35).sum()
    }

print(f"\nDone! {len(all_cleaned_dfs)} countries processed.")


PROCESSING: Ethiopia


FileNotFoundError: [Errno 2] No such file or directory: '..\\ethiopia.csv'

In [ ]:
fig, axes = plt.subplots(len(all_cleaned_dfs), 1, figsize=(16, 4*len(all_cleaned_dfs)))
if len(all_cleaned_dfs) == 1:
    axes = [axes]

for idx, df in enumerate(all_cleaned_dfs):
    monthly = df.set_index('Date').resample('M')['T2M'].mean()
    axes[idx].plot(monthly.index, monthly.values, color='crimson', marker='.', linewidth=1.5)
    axes[idx].set_title(f'Monthly Mean Temperature - {df["Country"].iloc[0]}', fontweight='bold')
    axes[idx].set_ylabel('Temperature (C)')
    axes[idx].grid(True, alpha=0.3)
    
    x = np.arange(len(monthly))
    z = np.polyfit(x, monthly.values, 1)
    p = np.poly1d(z)
    axes[idx].plot(monthly.index, p(x), '--', color='gray', alpha=0.7, label=f'Trend: {z[0]*12:.3f}C/yr')
    axes[idx].legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(len(all_cleaned_dfs), 1, figsize=(16, 4*len(all_cleaned_dfs)))
if len(all_cleaned_dfs) == 1:
    axes = [axes]

for idx, df in enumerate(all_cleaned_dfs):
    monthly = df.set_index('Date').resample('M')['PRECTOTCORR'].sum()
    axes[idx].bar(monthly.index, monthly.values, color='steelblue', width=20)
    axes[idx].set_title(f'Monthly Total Precipitation - {df["Country"].iloc[0]}', fontweight='bold')
    axes[idx].set_ylabel('Precipitation (mm)')
    axes[idx].grid(True, alpha=0.3, axis='y')
    axes[idx].axhline(y=monthly.mean(), color='red', linestyle='--')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(all_cleaned_dfs), figsize=(6*len(all_cleaned_dfs), 5))
if len(all_cleaned_dfs) == 1:
    axes = [axes]

for idx, df in enumerate(all_cleaned_dfs):
    corr = df[NUMERIC_COLS].corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, cmap='RdBu_r', center=0, fmt='.2f', ax=axes[idx])
    axes[idx].set_title(df['Country'].iloc[0])

plt.suptitle('Correlation Matrices', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(len(all_cleaned_dfs), 2, figsize=(14, 4*len(all_cleaned_dfs)))
if len(all_cleaned_dfs) == 1:
    axes = axes.reshape(1, -1)

for idx, df in enumerate(all_cleaned_dfs):
    axes[idx, 0].scatter(df['T2M'], df['RH2M'], alpha=0.3, s=10, c='teal')
    axes[idx, 0].set_xlabel('T2M (C)')
    axes[idx, 0].set_ylabel('RH2M (%)')
    axes[idx, 0].set_title(f'T2M vs RH2M - {df["Country"].iloc[0]}')
    axes[idx, 0].grid(True, alpha=0.3)
    
    axes[idx, 1].scatter(df['T2M_RANGE'], df['WS2M'], alpha=0.3, s=10, c='coral')
    axes[idx, 1].set_xlabel('T2M_RANGE (C)')
    axes[idx, 1].set_ylabel('WS2M (m/s)')
    axes[idx, 1].set_title(f'T2M_RANGE vs WS2M - {df["Country"].iloc[0]}')
    axes[idx, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(len(all_cleaned_dfs), 1, figsize=(12, 4*len(all_cleaned_dfs)))
if len(all_cleaned_dfs) == 1:
    axes = [axes]

for idx, df in enumerate(all_cleaned_dfs):
    axes[idx].hist(df['PRECTOTCORR'].dropna(), bins=50, color='steelblue', edgecolor='white')
    axes[idx].set_yscale('log')
    axes[idx].set_xlabel('Precipitation (mm/day)')
    axes[idx].set_title(f'Precipitation Distribution - {df["Country"].iloc[0]}')
    axes[idx].axvline(df['PRECTOTCORR'].mean(), color='red', linestyle='--')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(all_cleaned_dfs), figsize=(6*len(all_cleaned_dfs), 5))
if len(all_cleaned_dfs) == 1:
    axes = [axes]

for idx, df in enumerate(all_cleaned_dfs):
    sample = df.sample(min(1000, len(df)))
    axes[idx].scatter(sample['T2M'], sample['RH2M'], s=sample['PRECTOTCORR']*20, 
                      c=sample['Month'], cmap='viridis', alpha=0.5)
    axes[idx].set_xlabel('T2M (C)')
    axes[idx].set_ylabel('RH2M (%)')
    axes[idx].set_title(df['Country'].iloc[0])

plt.suptitle('T2M vs RH2M (Bubble size = Precipitation)', fontweight='bold')
plt.tight_layout()
plt.show()